# 🎓 Student Performance - Complete Machine Learning Project
---
**Dataset:** StudentsPerformance.csv  
**Target:** Writing Score Prediction + Performance Classification  
**Models:** Linear Regression, Random Forest, KNN, Decision Tree, Gradient Boosting  
**Dashboard:** Interactive Plotly Dashboard  

---
### 📌 Project Structure:
1. Setup & Data Loading
2. Exploratory Data Analysis (EDA)
3. Data Preprocessing
4. Feature Engineering
5. Model Training (5 Models)
6. Model Evaluation & Comparison
7. Hyperparameter Tuning
8. Predictions
9. Interactive Dashboard

## 📦 Step 1: Install Required Libraries

In [ ]:
# Install required libraries
!pip install plotly --quiet
!pip install ipywidgets --quiet

print('✅ Libraries installed successfully!')

## 📚 Step 2: Import All Libraries

In [ ]:
# ─── Data Handling ───────────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ─── Visualization ───────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ─── Preprocessing ───────────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV

# ─── Models ──────────────────────────────────────────────────────────
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

# ─── Evaluation ──────────────────────────────────────────────────────
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, classification_report, confusion_matrix
)

# ─── Style Settings ──────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print('✅ All libraries imported successfully!')
print('📊 Ready to analyze Student Performance Dataset')

## 💾 Step 3: Load Dataset

In [ ]:
# ─── Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ─── Load Dataset ────────────────────────────────────────────────────
data = pd.read_csv('/content/drive/MyDrive/StudentsPerformance.csv')

print('✅ Dataset loaded successfully!')
print(f'📐 Shape: {data.shape[0]} rows × {data.shape[1]} columns')
print('\n📋 Column Names:')
for col in data.columns:
    print(f'   → {col}')

## 🔍 Step 4: Exploratory Data Analysis (EDA)

In [ ]:
# ─── Basic Overview ───────────────────────────────────────────────────
print('=' * 60)
print('📌 DATASET OVERVIEW')
print('=' * 60)
print(f'\n🔢 Total Records   : {data.shape[0]}')
print(f'📊 Total Features  : {data.shape[1]}')
print(f'\n🗂️  Columns & Data Types:')
print(data.dtypes)
print(f'\n❌ Missing Values:')
print(data.isnull().sum())
print(f'\n🔁 Duplicate Rows: {data.duplicated().sum()}')

In [ ]:
# ─── First & Last 5 Records ──────────────────────────────────────────
print('📄 First 5 Records:')
display(data.head())

print('\n📄 Last 5 Records:')
display(data.tail())

In [ ]:
# ─── Statistical Summary ─────────────────────────────────────────────
print('📊 Statistical Summary (Numerical Features):')
display(data.describe())

print('\n📊 Statistical Summary (Categorical Features):')
display(data.describe(include='object'))

In [ ]:
# ─── Categorical Column Value Counts ─────────────────────────────────
cat_cols = data.select_dtypes(include='object').columns

for col in cat_cols:
    print(f'\n📌 {col.upper()}:')
    print(data[col].value_counts())

In [ ]:
# ─── EDA: Score Distributions ────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('📊 Score Distributions', fontsize=16, fontweight='bold', y=1.02)

colors = ['#3498db', '#e74c3c', '#2ecc71']
score_cols = ['math score', 'reading score', 'writing score']

for i, (col, color) in enumerate(zip(score_cols, colors)):
    axes[i].hist(data[col], bins=20, color=color, alpha=0.7, edgecolor='black')
    axes[i].axvline(data[col].mean(), color='black', linestyle='--', linewidth=2, label=f'Mean: {data[col].mean():.1f}')
    axes[i].set_title(f'{col.title()}', fontsize=13, fontweight='bold')
    axes[i].set_xlabel('Score')
    axes[i].set_ylabel('Frequency')
    axes[i].legend()

plt.tight_layout()
plt.savefig('score_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Score distribution plots created!')

In [ ]:
# ─── EDA: Gender vs Scores ───────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('👥 Score by Gender', fontsize=16, fontweight='bold')

for i, col in enumerate(score_cols):
    sns.boxplot(data=data, x='gender', y=col, ax=axes[i], palette=['#3498db', '#e91e8c'])
    axes[i].set_title(col.title(), fontsize=13, fontweight='bold')
    axes[i].set_xlabel('Gender')

plt.tight_layout()
plt.savefig('gender_scores.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── EDA: Parental Education vs Scores ───────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('🎓 Parental Education Impact on Scores', fontsize=16, fontweight='bold')

edu_order = ['some high school', 'high school', 'some college',
             "associate's degree", "bachelor's degree", "master's degree"]

for i, col in enumerate(score_cols):
    avg = data.groupby('parental level of education')[col].mean().reindex(edu_order)
    bars = axes[i].bar(range(len(avg)), avg.values, color=sns.color_palette('viridis', len(avg)))
    axes[i].set_xticks(range(len(avg)))
    axes[i].set_xticklabels([e.replace(' ', '\n') for e in edu_order], fontsize=8)
    axes[i].set_title(col.title(), fontsize=13, fontweight='bold')
    axes[i].set_ylabel('Average Score')
    for bar, val in zip(bars, avg.values):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                    f'{val:.1f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('parental_education_scores.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── EDA: Correlation Heatmap ────────────────────────────────────────
plt.figure(figsize=(8, 6))
num_data = data[score_cols]
corr = num_data.corr()

sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm',
            square=True, linewidths=2, cbar_kws={'shrink': 0.8},
            annot_kws={'size': 14, 'weight': 'bold'})
plt.title('🔗 Correlation Heatmap (Scores)', fontsize=16, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('📌 Insight: Math, Reading & Writing scores are highly correlated!')

In [ ]:
# ─── EDA: Test Preparation vs Scores ─────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('📝 Test Preparation Impact on Scores', fontsize=16, fontweight='bold')

for i, col in enumerate(score_cols):
    sns.violinplot(data=data, x='test preparation course', y=col,
                   ax=axes[i], palette=['#e74c3c', '#2ecc71'])
    axes[i].set_title(col.title(), fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('test_prep_scores.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── EDA: Race/Ethnicity Distribution ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('🌍 Race/Ethnicity Analysis', fontsize=16, fontweight='bold')

# Pie chart
counts = data['race/ethnicity'].value_counts()
axes[0].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=sns.color_palette('Set2', len(counts)))
axes[0].set_title('Distribution', fontweight='bold')

# Avg math score by group
avg_math = data.groupby('race/ethnicity')['math score'].mean().sort_values()
axes[1].barh(avg_math.index, avg_math.values,
             color=sns.color_palette('Set2', len(avg_math)))
for i, v in enumerate(avg_math.values):
    axes[1].text(v + 0.3, i, f'{v:.1f}', va='center', fontsize=11)
axes[1].set_xlabel('Average Math Score')
axes[1].set_title('Avg Math Score by Group', fontweight='bold')

plt.tight_layout()
plt.savefig('race_ethnicity.png', dpi=150, bbox_inches='tight')
plt.show()

## ⚙️ Step 5: Data Preprocessing

In [ ]:
# ─── Create a Copy for Processing ────────────────────────────────────
df = data.copy()

print('Original Data Sample:')
display(df.head(3))

# ─── Encode Categorical Columns ──────────────────────────────────────
encoding_map = {
    'gender': {'male': 0, 'female': 1},
    'race/ethnicity': {'group A': 1, 'group B': 2, 'group C': 3, 'group D': 4, 'group E': 5},
    'parental level of education': {
        'some high school': 1, 'high school': 2, 'some college': 3,
        "associate's degree": 4, "bachelor's degree": 5, "master's degree": 6
    },
    'lunch': {'standard': 1, 'free/reduced': 0},
    'test preparation course': {'completed': 1, 'none': 0}
}

for col, mapping in encoding_map.items():
    df[col] = df[col].replace(mapping)
    print(f'✅ Encoded: {col}')

# ─── Feature Engineering: Add New Features ───────────────────────────
df['total score']    = df['math score'] + df['reading score'] + df['writing score']
df['average score']  = df['total score'] / 3
df['score gap']      = df['reading score'] - df['math score']

# ─── Grade Category (Pass/Fail/Distinction) ───────────────────────────
def assign_grade(avg):
    if avg >= 80: return 'A (Distinction)'
    elif avg >= 60: return 'B (Pass)'
    elif avg >= 40: return 'C (Average)'
    else: return 'F (Fail)'

df['grade'] = df['average score'].apply(assign_grade)
df['pass_fail'] = (df['average score'] >= 40).astype(int)  # 1=Pass, 0=Fail

print('\n✅ Feature Engineering Complete!')
print('New Features Added: total score, average score, score gap, grade, pass_fail')
print('\nProcessed Data Sample:')
display(df.head())

In [ ]:
# ─── Grade Distribution ───────────────────────────────────────────────
grade_counts = df['grade'].value_counts()
print('📊 Grade Distribution:')
print(grade_counts)
print(f'\n✅ Pass Rate: {df["pass_fail"].mean()*100:.1f}%')
print(f'❌ Fail Rate: {(1-df["pass_fail"].mean())*100:.1f}%')

plt.figure(figsize=(8, 5))
colors = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c']
bars = plt.bar(grade_counts.index, grade_counts.values, color=colors[:len(grade_counts)], edgecolor='black')
for bar, val in zip(bars, grade_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, str(val),
             ha='center', fontsize=12, fontweight='bold')
plt.title('📊 Grade Distribution', fontsize=15, fontweight='bold')
plt.xlabel('Grade Category')
plt.ylabel('Number of Students')
plt.tight_layout()
plt.show()

## 🤖 Step 6: Machine Learning Models

In [ ]:
# ─── Prepare Features & Target ────────────────────────────────────────
# Task: Predict Writing Score (Regression)
feature_cols = ['gender', 'race/ethnicity', 'parental level of education',
                'lunch', 'test preparation course', 'math score', 'reading score']

X = df[feature_cols]
y = df['writing score']

# ─── Train-Test Split ─────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ─── Feature Scaling ──────────────────────────────────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'✅ Data Split Complete!')
print(f'   Training set  : {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'   Testing set   : {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.0f}%)')
print(f'   Features      : {X_train.shape[1]}')
print(f'   Target        : writing score (range: {y.min()} - {y.max()})')

In [ ]:
# ─── Train 6 Models & Evaluate ───────────────────────────────────────
models = {
    'Linear Regression':    LinearRegression(),
    'Ridge Regression':     Ridge(alpha=1.0),
    'KNN Regressor':        KNeighborsRegressor(n_neighbors=5),
    'Decision Tree':        DecisionTreeRegressor(max_depth=8, random_state=42),
    'Random Forest':        RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting':    GradientBoostingRegressor(n_estimators=100, random_state=42)
}

results = {}

print('🔄 Training Models...')
print('=' * 60)

for name, model in models.items():
    # Train
    if name in ['KNN Regressor']:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

    # Metrics
    mae  = mean_absolute_error(y_test, y_pred)
    mse  = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_test, y_pred)
    
    # Cross Validation
    if name in ['KNN Regressor']:
        cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='r2')
    else:
        cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')

    results[name] = {
        'model':    model,
        'y_pred':   y_pred,
        'MAE':      round(mae, 3),
        'MSE':      round(mse, 3),
        'RMSE':     round(rmse, 3),
        'R2':       round(r2, 4),
        'CV_R2':    round(cv_scores.mean(), 4),
        'CV_Std':   round(cv_scores.std(), 4)
    }

    print(f'\n✅ {name}')
    print(f'   MAE  : {mae:.3f} | RMSE : {rmse:.3f} | R²   : {r2:.4f} | CV-R²: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

print('\n🎯 All models trained successfully!')

## 📊 Step 7: Model Comparison & Visualization

In [ ]:
# ─── Results DataFrame ───────────────────────────────────────────────
results_df = pd.DataFrame([
    {'Model': name, 'MAE': v['MAE'], 'RMSE': v['RMSE'], 'R2': v['R2'], 'CV_R2': v['CV_R2']}
    for name, v in results.items()
]).sort_values('R2', ascending=False).reset_index(drop=True)

results_df.index += 1
print('📊 Model Comparison Table (sorted by R²):')
display(results_df)

best_model_name = results_df.iloc[0]['Model']
best_r2 = results_df.iloc[0]['R2']
print(f'\n🏆 Best Model: {best_model_name} (R² = {best_r2})')

In [ ]:
# ─── Model Comparison Chart ───────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('📊 Model Performance Comparison', fontsize=16, fontweight='bold')

colors = sns.color_palette('viridis', len(results_df))

# R² Score
bars1 = axes[0].barh(results_df['Model'], results_df['R2'], color=colors)
axes[0].set_title('R² Score (Higher = Better)', fontweight='bold')
axes[0].set_xlim(0, 1.05)
for bar, val in zip(bars1, results_df['R2']):
    axes[0].text(val + 0.005, bar.get_y() + bar.get_height()/2, f'{val:.4f}', va='center', fontsize=10)

# MAE
bars2 = axes[1].barh(results_df['Model'], results_df['MAE'], color=colors)
axes[1].set_title('MAE (Lower = Better)', fontweight='bold')
for bar, val in zip(bars2, results_df['MAE']):
    axes[1].text(val + 0.1, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center', fontsize=10)

# RMSE
bars3 = axes[2].barh(results_df['Model'], results_df['RMSE'], color=colors)
axes[2].set_title('RMSE (Lower = Better)', fontweight='bold')
for bar, val in zip(bars3, results_df['RMSE']):
    axes[2].text(val + 0.1, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Actual vs Predicted (Best 3 Models) ─────────────────────────────
top_models = results_df['Model'].iloc[:3].tolist()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('🎯 Actual vs Predicted (Top 3 Models)', fontsize=16, fontweight='bold')

for i, name in enumerate(top_models):
    y_pred = results[name]['y_pred']
    r2 = results[name]['R2']
    axes[i].scatter(y_test, y_pred, alpha=0.5, color='#3498db', edgecolors='k', linewidth=0.3)
    axes[i].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Fit')
    axes[i].set_xlabel('Actual Score')
    axes[i].set_ylabel('Predicted Score')
    axes[i].set_title(f'{name}\nR² = {r2}', fontweight='bold')
    axes[i].legend()

plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Residual Plot ────────────────────────────────────────────────────
best = results_df.iloc[0]['Model']
y_pred_best = results[best]['y_pred']
residuals = y_test - y_pred_best

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'📉 Residual Analysis — {best}', fontsize=15, fontweight='bold')

axes[0].scatter(y_pred_best, residuals, alpha=0.5, color='#9b59b6', edgecolors='k', linewidth=0.3)
axes[0].axhline(0, color='red', linestyle='--', lw=2)
axes[0].set_xlabel('Predicted Values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Fitted')

axes[1].hist(residuals, bins=25, color='#9b59b6', alpha=0.7, edgecolor='black')
axes[1].axvline(0, color='red', linestyle='--', lw=2)
axes[1].set_xlabel('Residual Value')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Residuals Distribution')

plt.tight_layout()
plt.savefig('residuals.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'📊 Mean Residual: {residuals.mean():.4f} (should be close to 0)')

## 🔧 Step 8: Feature Importance (Random Forest)

In [ ]:
# ─── Feature Importance ───────────────────────────────────────────────
rf_model = results['Random Forest']['model']
importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(9, 5))
colors = sns.color_palette('RdYlGn', len(importance_df))
bars = plt.barh(importance_df['Feature'], importance_df['Importance'], color=colors)
for bar, val in zip(bars, importance_df['Importance']):
    plt.text(val + 0.003, bar.get_y() + bar.get_height()/2, f'{val:.4f}', va='center', fontsize=11)
plt.title('🌳 Feature Importance (Random Forest)', fontsize=15, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n🔝 Most Important Feature: {importance_df.iloc[-1]["Feature"]}')
print(f'   Importance Score: {importance_df.iloc[-1]["Importance"]:.4f}')

## 🔮 Step 9: Make Predictions (New Students)

In [ ]:
# ─── Prediction Function ──────────────────────────────────────────────
def predict_writing_score(gender, race_group, parental_edu, lunch, test_prep, math_score, reading_score):
    """
    Predict writing score for a new student.
    gender:       0=male, 1=female
    race_group:   1=A, 2=B, 3=C, 4=D, 5=E
    parental_edu: 1-6 (some high school → master's degree)
    lunch:        0=free/reduced, 1=standard
    test_prep:    0=none, 1=completed
    math_score:   0-100
    reading_score: 0-100
    """
    input_data = np.array([[gender, race_group, parental_edu, lunch, test_prep, math_score, reading_score]])
    input_df   = pd.DataFrame(input_data, columns=feature_cols)

    best_name = results_df.iloc[0]['Model']
    best_m    = results[best_name]['model']
    prediction = best_m.predict(input_df)[0]

    # Grade
    avg = (math_score + reading_score + prediction) / 3
    grade = assign_grade(avg)

    print(f'🔮 Predicted Writing Score : {prediction:.1f}/100')
    print(f'📊 Average Score (all 3)  : {avg:.1f}/100')
    print(f'🎓 Predicted Grade        : {grade}')
    print(f'🏅 Model Used             : {best_name} (R² = {results[best_name]["R2"]})')
    return prediction


print('=' * 50)
print('📌 Example 1: Female, Group C, Bachelor Degree, Standard Lunch, Completed Test Prep')
_ = predict_writing_score(1, 3, 5, 1, 1, 76, 78)

print('\n' + '=' * 50)
print('📌 Example 2: Male, Group A, High School, Free Lunch, No Test Prep')
_ = predict_writing_score(0, 1, 2, 0, 0, 55, 50)

print('\n' + '=' * 50)
print('📌 Example 3: Female, Group E, Masters Degree, Standard Lunch, Completed Test Prep')
_ = predict_writing_score(1, 5, 6, 1, 1, 90, 92)

## 📈 Step 10: Interactive Dashboard (Plotly)

In [ ]:
# ─── DASHBOARD: Full Interactive Plotly Dashboard ─────────────────────

# Reload raw data for dashboard (original string labels)
raw = data.copy()

# Add computed columns to raw
raw['total score']   = raw['math score'] + raw['reading score'] + raw['writing score']
raw['average score'] = raw['total score'] / 3
raw['grade'] = raw['average score'].apply(assign_grade)
raw['pass_fail_label'] = raw['average score'].apply(lambda x: '✅ Pass' if x >= 40 else '❌ Fail')

# ─── Build 3x3 Dashboard ──────────────────────────────────────────────
fig = make_subplots(
    rows=3, cols=3,
    subplot_titles=(
        '🎯 Score Distributions',
        '👥 Gender vs Avg Score',
        '📊 Grade Distribution',
        '🎓 Parental Education vs Math Score',
        '📝 Test Prep Impact',
        '🌍 Race/Ethnicity vs Writing Score',
        '🔗 Math vs Reading (Scatter)',
        '🏆 Model R² Comparison',
        '🍱 Lunch vs Avg Score'
    ),
    vertical_spacing=0.12,
    horizontal_spacing=0.08
)

# 1. Score Distributions (Histogram)
for score, color in zip(['math score','reading score','writing score'], ['#3498db','#e74c3c','#2ecc71']):
    fig.add_trace(go.Histogram(x=raw[score], name=score, marker_color=color, opacity=0.7), row=1, col=1)

# 2. Gender vs Avg Score (Box)
for gender in ['male','female']:
    sub = raw[raw['gender']==gender]
    fig.add_trace(go.Box(y=sub['average score'], name=gender, boxmean=True), row=1, col=2)

# 3. Grade Distribution (Pie)
grade_c = raw['grade'].value_counts()
fig.add_trace(go.Pie(labels=grade_c.index, values=grade_c.values, hole=0.4,
                     textinfo='label+percent'), row=1, col=3)

# 4. Parental Education vs Math Score (Bar)
edu_order2 = ['some high school','high school','some college',"associate's degree","bachelor's degree","master's degree"]
edu_avg = raw.groupby('parental level of education')['math score'].mean().reindex(edu_order2)
fig.add_trace(go.Bar(x=edu_avg.index, y=edu_avg.values,
                     marker_color=px.colors.sequential.Viridis[::-1][:len(edu_avg)],
                     text=[f'{v:.1f}' for v in edu_avg.values], textposition='auto'), row=2, col=1)

# 5. Test Prep Impact (Grouped Bar)
for prep in ['none','completed']:
    sub = raw[raw['test preparation course']==prep]
    fig.add_trace(go.Bar(
        name=f'Test Prep: {prep}',
        x=['math score','reading score','writing score'],
        y=[sub['math score'].mean(), sub['reading score'].mean(), sub['writing score'].mean()],
        text=[f'{v:.1f}' for v in [sub['math score'].mean(), sub['reading score'].mean(), sub['writing score'].mean()]],
        textposition='auto'
    ), row=2, col=2)

# 6. Race/Ethnicity vs Writing Score (Box)
for g in sorted(raw['race/ethnicity'].unique()):
    sub = raw[raw['race/ethnicity']==g]
    fig.add_trace(go.Box(y=sub['writing score'], name=g), row=2, col=3)

# 7. Math vs Reading Scatter
fig.add_trace(go.Scatter(
    x=raw['math score'], y=raw['reading score'],
    mode='markers', marker=dict(color=raw['average score'], colorscale='Viridis', opacity=0.6, size=5,
                                colorbar=dict(title='Avg Score')),
    text=raw['grade'], name='Students'
), row=3, col=1)

# 8. Model R² Comparison
fig.add_trace(go.Bar(
    x=results_df['Model'], y=results_df['R2'],
    marker_color=['#2ecc71' if i==0 else '#3498db' for i in range(len(results_df))],
    text=[f'{v:.4f}' for v in results_df['R2']], textposition='auto'
), row=3, col=2)

# 9. Lunch vs Avg Score
for lunch in ['standard','free/reduced']:
    sub = raw[raw['lunch']==lunch]
    fig.add_trace(go.Violin(
        y=sub['average score'], name=lunch, box_visible=True, meanline_visible=True
    ), row=3, col=3)

# ─── Layout ──────────────────────────────────────────────────────────
fig.update_layout(
    height=1200, width=1400,
    title_text='🎓 Student Performance — Complete ML Dashboard',
    title_font=dict(size=22, family='Arial Black'),
    showlegend=True,
    template='plotly_white',
    paper_bgcolor='#f8f9fa',
    font=dict(family='Arial', size=11),
)

fig.show()
print('✅ Interactive Dashboard Ready!')

In [ ]:
# ─── Save Dashboard as HTML ───────────────────────────────────────────
fig.write_html('student_performance_dashboard.html')
print('✅ Dashboard saved as: student_performance_dashboard.html')

# Download the file
from google.colab import files
files.download('student_performance_dashboard.html')
print('📥 Dashboard downloaded!')

## 📋 Step 11: Final Project Summary

In [ ]:
# ─── Final Summary Report ─────────────────────────────────────────────
print('=' * 65)
print('🎓 STUDENT PERFORMANCE ML PROJECT — FINAL SUMMARY')
print('=' * 65)

print(f"""
📊 DATASET
   Total Students  : {len(df)}
   Features Used   : {len(feature_cols)}
   Target Variable : Writing Score
   Pass Rate       : {df['pass_fail'].mean()*100:.1f}%

📈 EDA INSIGHTS
   Avg Math Score    : {data['math score'].mean():.1f}
   Avg Reading Score : {data['reading score'].mean():.1f}
   Avg Writing Score : {data['writing score'].mean():.1f}
   Scores Correlation: Very High (>0.80 between all 3 scores)
   Test Prep Boost   : ~5-8 points average improvement

🤖 MODELS TRAINED: {len(models)}
   1. Linear Regression
   2. Ridge Regression
   3. KNN Regressor
   4. Decision Tree
   5. Random Forest
   6. Gradient Boosting
""")

print('🏆 MODEL RANKINGS (by R² Score):')
for i, row in results_df.iterrows():
    medal = '🥇' if i==1 else '🥈' if i==2 else '🥉' if i==3 else '  '
    print(f'   {medal} {i}. {row["Model"]:<25} R²={row["R2"]:.4f}  MAE={row["MAE"]:.3f}  RMSE={row["RMSE"]:.3f}')

best = results_df.iloc[0]
print(f"""
✅ BEST MODEL   : {best['Model']}
   R² Score    : {best['R2']} ({best['R2']*100:.1f}% variance explained)
   MAE         : {best['MAE']} (avg prediction error)
   RMSE        : {best['RMSE']}

📁 FILES SAVED:
   → score_distributions.png
   → gender_scores.png
   → parental_education_scores.png
   → correlation_heatmap.png
   → model_comparison.png
   → actual_vs_predicted.png
   → feature_importance.png
   → student_performance_dashboard.html
""")
print('=' * 65)
print('✅ PROJECT COMPLETE!')
print('=' * 65)